# OmniOpinion: Two-Timescale Conviction vs. Current Opinion Dynamics in a 100k City ($N = 100{,}000$, $T = 1{,}500$ Steps)

This notebook analyzes the **two-timescale attitude dynamics** in a medium-sized city ($N = 100{,}000$ citizens, 20 districts, $M=3$ tripolar traditional press + multi-platform social media) over $T = 1{,}500$ steps:

1. **Convictions ($\mathbf{s}_i(t) \in [-1, 1]^4$) vs. Current Opinions ($\mathbf{x}_i(t) \in [-1, 1]^4$):**
   - **Convictions** represent deep ideological anchors (`innate_beliefs`) that evolve slowly ($\eta_{\mathrm{conv}} \ll \eta_{\mathrm{op}}$) via exponential crystallization ($\tau_c = 280$ steps) and demographic inertia.
   - **Current Opinions** represent day-to-day expressed attitudes (`opinions`) that directly drive electoral voting (`ElectoralEngine`) and respond rapidly to peer interactions, media narratives, and objective news facts, while remaining **elastically tethered** within a citizen-specific radius $\Delta_{\max, i}$ around their conviction $\mathbf{s}_i(t)$.
2. **Conviction Stabilization vs. Electoral Opinion Mobility:**
   - Within the 100k city, convictions crystallize and stabilize (`conviction_step_volatility` $\to 0$), whereas current opinions continue to fluctuate (`opinion_step_volatility` $\gg$ `conviction_step_volatility`) within the conviction tether (`mean_op_conv_gap`), causing a substantial fraction of citizens (`swing_voter_fraction`) to vote for a different candidate than their underlying conviction alone would predict (`election_flipped_by_opinion`).
3. **Dependence on Citizen Features & Centrist vs. Extremist Convictions:**
   - Breakdown across **Age cohorts** (`Young <30`, `Mid 30-54`, `Senior >=55`), **Socioeconomic/Educational Status** (`Low SES <0.35`, `High SES >=0.55`), and **Conviction Extremity** ($E_i = \|\mathbf{s}_i\|_2 / \sqrt{D}$: `Centrist <0.35`, `Moderate 0.35-0.60`, `Extremist >0.60`).
   - Comparison between a **Centrist Conviction City** (`centrist_convictions`), a **Uniform Baseline City** (`uniform_random`), and an **Extremist Conviction City** (`extremist_convictions`).
4. **Strong Regime Shift in the Bias of Exogenous Facts (`fact_bias_mode = "regime_shift"`):**
   - Phase 1 ($t \in [0, 400]$): Neutral objective facts ($\mathbf{b}_{\mathrm{fact}} = 0.0$).
   - Phase 2 ($t \in [401, 950]$): Strong positive shift in fact bias ($b_{\mathrm{fact}, 0} = +0.78$).
   - Phase 3 ($t \in [951, 1500]$): Strong negative reversal in fact bias ($b_{\mathrm{fact}, 0} = -0.82$).
   - Demonstrates two-timescale hysteresis: current opinions $\bar{x}_0(t)$ shift rapidly with the biased facts and flip election winners, whereas convictions $\bar{s}_0(t)$ drift much more slowly.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.15)
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 300

SCRATCH_CONSOLIDATED = Path('/scratch/global/crcosta/omni_opinion/consolidated/all_conviction_dynamics_metrics.parquet')
LOCAL_CONSOLIDATED = RESULTS_DIR / 'all_conviction_dynamics_metrics.parquet'
data_path = SCRATCH_CONSOLIDATED if SCRATCH_CONSOLIDATED.exists() else LOCAL_CONSOLIDATED

df = pd.read_parquet(data_path)
if 'scenario_name' not in df.columns and 'scenario' in df.columns:
    df['scenario_name'] = df['scenario']
print(f'Loaded conviction-opinion dataset from {data_path.name}: {len(df):,} rows')
print('Scenarios:', sorted(df['scenario_name'].unique()))
print('Steps range:', df['step'].min(), 'to', df['step'].max(), '| Seeds per scenario:', df.groupby('scenario_name')['task_id'].nunique().to_dict())

Loaded conviction-opinion dataset from all_conviction_dynamics_metrics.parquet: 9,030 rows
Scenarios: ['conv_centrist_factshift', 'conv_centrist_neutral', 'conv_extremist_factshift', 'conv_extremist_neutral', 'conv_uniform_factshift', 'conv_uniform_neutral']
Steps range: 0 to 1500 | Seeds per scenario: {'conv_centrist_factshift': 5, 'conv_centrist_neutral': 5, 'conv_extremist_factshift': 5, 'conv_extremist_neutral': 5, 'conv_uniform_factshift': 5, 'conv_uniform_neutral': 5}


## 1. How Convictions Stabilize While Current Opinions Fluctuate and Alter Elections

We first inspect the three neutral-fact city baselines (`conv_uniform_neutral`, `conv_centrist_neutral`, `conv_extremist_neutral`) to compare:
- **Step-to-step volatility:** `opinion_step_volatility` ($\frac{1}{N}\sum_i \|\mathbf{x}_i(t) - \mathbf{x}_i(t-1)\|_2 / \sqrt{D}$) vs. `conviction_step_volatility` ($\frac{1}{N}\sum_i \|\mathbf{s}_i(t) - \mathbf{s}_i(t-1)\|_2 / \sqrt{D}$).
- **Bounded opinion-conviction gap:** `mean_op_conv_gap` ($\frac{1}{N}\sum_i \|\mathbf{x}_i(t) - \mathbf{s}_i(t)\|_2 / \sqrt{D}$), showing that opinions move dynamically around convictions without breaking their tether.
- **Dispersion stabilization:** `conv_dispersion_ratio` stabilizing as convictions crystallize while `dispersion_ratio_vs_random` reflects active opinion clustering.
- **Electoral impact (`swing_voter_fraction` and `election_flipped_by_opinion`):** The percentage of voters whose current opinion ballot (`popular_winner`) differs from their underlying conviction ballot (`conv_popular_winner`).

In [2]:
neutral_scens = ['conv_uniform_neutral', 'conv_centrist_neutral', 'conv_extremist_neutral']
df_neutral = df[df['scenario_name'].isin(neutral_scens)].copy()

scen_labels = {
    'conv_uniform_neutral': 'Uniform City (Baseline)',
    'conv_centrist_neutral': 'Centrist Conviction City',
    'conv_extremist_neutral': 'Extremist Conviction City',
}
palette_city = {
    'conv_uniform_neutral': '#1f77b4',
    'conv_centrist_neutral': '#2ca02c',
    'conv_extremist_neutral': '#d62728',
}

final_neutral = df_neutral[df_neutral['step'] == df_neutral['step'].max()]
summary_cols = [
    'opinion_step_volatility',
    'conviction_step_volatility',
    'mean_op_conv_gap',
    'conv_dispersion_ratio',
    'dispersion_ratio_vs_random',
    'swing_voter_fraction',
    'election_flipped_by_opinion',
]
summary_table = final_neutral.groupby('scenario_name')[summary_cols].mean().round(4)
summary_table['vol_ratio_op_to_conv'] = (
    summary_table['opinion_step_volatility'] / summary_table['conviction_step_volatility'].clip(lower=1e-6)
).round(1)
print('=== Final-Step (t=1500) Conviction Stabilization vs. Opinion Mobility Summary ===')
print(summary_table.to_string())

=== Final-Step (t=1500) Conviction Stabilization vs. Opinion Mobility Summary ===
                        opinion_step_volatility  conviction_step_volatility  mean_op_conv_gap  conv_dispersion_ratio  dispersion_ratio_vs_random  swing_voter_fraction  election_flipped_by_opinion  vol_ratio_op_to_conv
scenario_name                                                                                                                                                                                                            
conv_centrist_neutral                    0.0077                      0.0000            0.0863                 0.0267                      0.0033                0.3301                          0.0                7700.0
conv_extremist_neutral                   0.0094                      0.0000            0.0865                 1.3605                      1.4174                0.1014                          0.4                9400.0
conv_uniform_neutral                     0.008

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10.5), sharex=True)

# Panel A: Step-to-Step Volatility (Opinion vs Conviction in log/linear scale)
ax = axes[0, 0]
for scen in neutral_scens:
    sub = df_neutral[df_neutral['scenario_name'] == scen]
    grp = sub.groupby('step')[['opinion_step_volatility', 'conviction_step_volatility']].mean()
    ax.plot(grp.index, grp['opinion_step_volatility'], color=palette_city[scen], lw=2.2,
            label=f"{scen_labels[scen]} — Current Opinion $\\Delta x$")
    ax.plot(grp.index, grp['conviction_step_volatility'], color=palette_city[scen], lw=2.0, ls='--',
            label=f"{scen_labels[scen]} — Conviction $\\Delta s$")
ax.set_yscale('log')
ax.set_title('A. Two-Timescale Step Volatility: Convictions Crystallize While Opinions Fluctuate', fontweight='bold')
ax.set_ylabel('Mean Step Volatility (Log Scale)')
ax.legend(fontsize=8.5, ncol=1, loc='upper right')

# Panel B: Bounded Opinion-Conviction Excursion Gap ||x_i(t) - s_i(t)||
ax = axes[0, 1]
for scen in neutral_scens:
    sub = df_neutral[df_neutral['scenario_name'] == scen]
    grp = sub.groupby('step')['mean_op_conv_gap'].agg(['mean', 'std'])
    ax.plot(grp.index, grp['mean'], color=palette_city[scen], lw=2.2, label=scen_labels[scen])
    ax.fill_between(grp.index, grp['mean'] - grp['std'], grp['mean'] + grp['std'], color=palette_city[scen], alpha=0.18)
ax.set_title('B. Bounded Excursion Gap Between Current Opinion and Conviction ($\\|\\mathbf{x}_i - \\mathbf{s}_i\\|$)', fontweight='bold')
ax.set_ylabel('Mean Opinion–Conviction Distance')
ax.legend(fontsize=9.5, loc='lower right')

# Panel C: Conviction Dispersion vs Current Opinion Dispersion Ratio
ax = axes[1, 0]
for scen in neutral_scens:
    sub = df_neutral[df_neutral['scenario_name'] == scen]
    grp = sub.groupby('step')[['conv_dispersion_ratio', 'dispersion_ratio_vs_random']].mean()
    ax.plot(grp.index, grp['conv_dispersion_ratio'], color=palette_city[scen], lw=2.2, ls='--',
            label=f"{scen_labels[scen]} — Conviction Dispersion")
    ax.plot(grp.index, grp['dispersion_ratio_vs_random'], color=palette_city[scen], lw=2.2,
            label=f"{scen_labels[scen]} — Opinion Dispersion")
ax.set_title('C. Structural Stabilization of Convictions vs. Active Clustering of Opinions', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Dispersion Ratio vs. Uniform Random')
ax.legend(fontsize=8.5, loc='upper right')

# Panel D: Electoral Swing Voters Driven by Opinion Mobility Around Convictions
ax = axes[1, 1]
for scen in neutral_scens:
    sub = df_neutral[df_neutral['scenario_name'] == scen]
    grp = sub.groupby('step')['swing_voter_fraction'].agg(['mean', 'std'])
    ax.plot(grp.index, 100.0 * grp['mean'], color=palette_city[scen], lw=2.3, label=scen_labels[scen])
    ax.fill_between(grp.index, 100.0 * (grp['mean'] - grp['std']), 100.0 * (grp['mean'] + grp['std']),
                    color=palette_city[scen], alpha=0.18)
ax.set_title('D. Electoral Impact: Voters Casting a Different Ballot Than Their Conviction Alone', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Swing Voter Percentage (%)')
ax.legend(fontsize=9.5, loc='lower right')

plt.tight_layout()
fig1_path = FIGURES_DIR / 'fig_cv1_conviction_stabilization_vs_opinion_mobility.png'
fig.savefig(fig1_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig1_path}')

<Figure size 2100x1470 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_cv1_conviction_stabilization_vs_opinion_mobility.png


## 2. How Opinion Mobility & Electoral Swing Depend on Person Features and Centrist vs. Extremist Convictions

Next, we examine how a citizen's demographic and ideological attributes shape the interplay between conviction and current opinion:
- **Age Cohorts (Life-Stage Plasticity vs. Crystallized Attitude Persistence):**
  - `Young (<30)`: Higher social media exposure and openness, wider conviction tether $\Delta_{\max, i}$, larger opinion excursion, and higher electoral swing rate.
  - `Mid-Age (30–54)`: Intermediate tether and swing rate.
  - `Senior (>=55)`: Strongly crystallized convictions, tighter tether $\Delta_{\max, i}$, and lower opinion volatility.
- **Socioeconomic / Educational Status (SES):**
  - `Low SES (<0.35)` vs. `High SES (>=0.55)`: Higher cognitive/educational resources provide stronger ideological constraint, keeping daily opinions closer to underlying convictions.
- **Conviction Extremity ($E_i = \|\mathbf{s}_i\|_2 / \sqrt{D}$):**
  - `Centrist Conviction (<0.35)`: Weak ego-involvement allows wider opinion mobility (`centrist_op_conv_gap`) and makes centrist citizens the decisive electoral swing group (`centrist_swing_vote`).
  - `Moderate Conviction (0.35–0.60)`: Intermediate mobility.
  - `Extremist Conviction (>0.60)`: High ego-involvement (Sherif & Hovland, 1961) strongly anchors daily opinions near the extreme conviction (`extremist_op_conv_gap`) and virtually locks in their vote (`extremist_swing_vote` is minimal).

In [4]:
steady_df = df_neutral[df_neutral['step'] >= 500].copy()
uni_steady = steady_df[steady_df['scenario_name'] == 'conv_uniform_neutral']

cohort_table = pd.DataFrame({
    'Group Category': [
        'Age: Young (<30)', 'Age: Mid (30-54)', 'Age: Senior (>=55)',
        'SES: Low (<0.35)', 'SES: High (>=0.55)',
        'Conviction: Centrist (<0.35)', 'Conviction: Moderate (0.35-0.60)', 'Conviction: Extremist (>0.60)'
    ],
    'Mean Opinion-Conviction Gap': [
        uni_steady['op_conv_gap_young'].mean(),
        uni_steady['op_conv_gap_mid'].mean(),
        uni_steady['op_conv_gap_senior'].mean(),
        uni_steady['op_conv_gap_low_ses'].mean(),
        uni_steady['op_conv_gap_high_ses'].mean(),
        uni_steady['op_conv_gap_centrist'].mean(),
        uni_steady['op_conv_gap_moderate'].mean(),
        uni_steady['op_conv_gap_extremist'].mean(),
    ],
    'Mean Opinion Step Volatility': [
        uni_steady['op_vol_young'].mean(),
        uni_steady['op_vol_mid'].mean(),
        uni_steady['op_vol_senior'].mean(),
        uni_steady['op_vol_low_ses'].mean(),
        uni_steady['op_vol_high_ses'].mean(),
        uni_steady['op_vol_centrist'].mean(),
        uni_steady['op_vol_moderate'].mean(),
        uni_steady['op_vol_extremist'].mean(),
    ],
    'Swing Vote Rate (%)': [
        100.0 * uni_steady['swing_vote_young'].mean(),
        100.0 * uni_steady['swing_vote_mid'].mean(),
        100.0 * uni_steady['swing_vote_senior'].mean(),
        100.0 * uni_steady['swing_vote_low_ses'].mean(),
        100.0 * uni_steady['swing_vote_high_ses'].mean(),
        100.0 * uni_steady['swing_vote_centrist'].mean(),
        100.0 * uni_steady['swing_vote_moderate'].mean(),
        100.0 * uni_steady['swing_vote_extremist'].mean(),
    ],
}).round(4)

print('=== Citizen Feature & Conviction Extremity Breakdown (Steady State t in [500, 1500]) ===')
print(cohort_table.to_string(index=False))

=== Citizen Feature & Conviction Extremity Breakdown (Steady State t in [500, 1500]) ===
                  Group Category  Mean Opinion-Conviction Gap  Mean Opinion Step Volatility  Swing Vote Rate (%)
                Age: Young (<30)                       0.1708                        0.0078              30.2630
                Age: Mid (30-54)                       0.1559                        0.0078              26.2447
              Age: Senior (>=55)                       0.1192                        0.0078              17.9124
                SES: Low (<0.35)                       0.1484                        0.0079              24.9690
              SES: High (>=0.55)                       0.1522                        0.0078              24.7010
    Conviction: Centrist (<0.35)                       0.1848                        0.0075              39.1863
Conviction: Moderate (0.35-0.60)                       0.1486                        0.0077              19.8436
   Conv

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(15.5, 10.5))

# Panel A: Time series of Opinion-Conviction Gap by Age & SES
ax = axes[0, 0]
uni_all = df_neutral[df_neutral['scenario_name'] == 'conv_uniform_neutral']
age_grp = uni_all.groupby('step')[[
    'op_conv_gap_young', 'op_conv_gap_mid', 'op_conv_gap_senior',
    'op_conv_gap_low_ses', 'op_conv_gap_high_ses'
]].mean()
ax.plot(age_grp.index, age_grp['op_conv_gap_young'], color='#ff7f0e', lw=2.3, label='Age: Young (<30)')
ax.plot(age_grp.index, age_grp['op_conv_gap_mid'], color='#1f77b4', lw=2.1, label='Age: Mid (30–54)')
ax.plot(age_grp.index, age_grp['op_conv_gap_senior'], color='#9467bd', lw=2.3, label='Age: Senior (>=55)')
ax.plot(age_grp.index, age_grp['op_conv_gap_low_ses'], color='#d62728', lw=1.9, ls='--', label='SES: Low (<0.35)')
ax.plot(age_grp.index, age_grp['op_conv_gap_high_ses'], color='#2ca02c', lw=1.9, ls='--', label='SES: High (>=0.55)')
ax.set_title('A. Opinion Excursion ($\\|\\mathbf{x}_i - \\mathbf{s}_i\\|$) by Age Cohort and SES', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Opinion–Conviction Distance')
ax.legend(fontsize=9, ncol=2, loc='lower right')

# Panel B: Time series of Opinion-Conviction Gap by Conviction Extremity (Centrist vs Moderate vs Extremist)
ax = axes[0, 1]
ext_grp = uni_all.groupby('step')[['op_conv_gap_centrist', 'op_conv_gap_moderate', 'op_conv_gap_extremist']].mean()
ax.plot(ext_grp.index, ext_grp['op_conv_gap_centrist'], color='#2ca02c', lw=2.5, label='Centrist Conviction ($E_i < 0.35$)')
ax.plot(ext_grp.index, ext_grp['op_conv_gap_moderate'], color='#1f77b4', lw=2.3, label='Moderate Conviction ($0.35 \\leq E_i \\leq 0.60$)')
ax.plot(ext_grp.index, ext_grp['op_conv_gap_extremist'], color='#d62728', lw=2.5, label='Extremist Conviction ($E_i > 0.60$)')
ax.set_title('B. Opinion Mobility Within Conviction Tether: Centrist vs. Extremist Citizens', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Opinion–Conviction Distance')
ax.legend(fontsize=9.5, loc='lower right')

# Panel C: Bar Chart of Electoral Swing Rate (%) Across Citizen Subgroups
ax = axes[1, 0]
subgroups = [
    'Young\n(<30)', 'Mid-Age\n(30–54)', 'Senior\n(>=55)',
    'Low SES\n(<0.35)', 'High SES\n(>=0.55)',
    'Centrist\nConviction', 'Moderate\nConviction', 'Extremist\nConviction'
]
swing_vals = [
    100.0 * uni_steady['swing_vote_young'].mean(),
    100.0 * uni_steady['swing_vote_mid'].mean(),
    100.0 * uni_steady['swing_vote_senior'].mean(),
    100.0 * uni_steady['swing_vote_low_ses'].mean(),
    100.0 * uni_steady['swing_vote_high_ses'].mean(),
    100.0 * uni_steady['swing_vote_centrist'].mean(),
    100.0 * uni_steady['swing_vote_moderate'].mean(),
    100.0 * uni_steady['swing_vote_extremist'].mean(),
]
bar_colors = ['#ff7f0e', '#1f77b4', '#9467bd', '#e377c2', '#17becf', '#2ca02c', '#7f7f7f', '#d62728']
bars = ax.bar(subgroups, swing_vals, color=bar_colors, edgecolor='black', alpha=0.88)
for b, v in zip(bars, swing_vals):
    ax.text(b.get_x() + b.get_width() / 2.0, v + 0.35, f'{v:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, max(swing_vals) * 1.18)
ax.set_title('C. Electoral Swing Rate (% Voting Differently Than Conviction) by Citizen Feature', fontweight='bold')
ax.set_ylabel('Swing Voters (%)')

# Panel D: City-Level Comparison (Centrist City vs Uniform City vs Extremist City)
ax = axes[1, 1]
city_metrics = steady_df.groupby('scenario_name')[[
    'mean_op_conv_gap', 'swing_voter_fraction', 'election_flipped_by_opinion'
]].mean().loc[neutral_scens]
x = np.arange(len(neutral_scens))
w = 0.26
b1 = ax.bar(x - w, 100.0 * city_metrics['mean_op_conv_gap'], width=w, color='#1f77b4', edgecolor='black',
            label='Mean Gap $\\|\\mathbf{x}-\\mathbf{s}\\| \\times 100$')
b2 = ax.bar(x, 100.0 * city_metrics['swing_voter_fraction'], width=w, color='#2ca02c', edgecolor='black',
            label='Swing Voter Share (%)')
b3 = ax.bar(x + w, 100.0 * city_metrics['election_flipped_by_opinion'], width=w, color='#d62728', edgecolor='black',
            label='Election Flipped by Opinion (%)')
ax.set_xticks(x)
ax.set_xticklabels(['Uniform City\n(Baseline)', 'Centrist\nConviction City', 'Extremist\nConviction City'])
ax.set_title('D. Macro-City Comparison: Centrist Conviction City vs. Extremist Conviction City', fontweight='bold')
ax.set_ylabel('Percentage / Scaled Index')
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
fig2_path = FIGURES_DIR / 'fig_cv2_person_features_and_centrist_vs_extremist.png'
fig.savefig(fig2_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig2_path}')

<Figure size 2170x1470 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_cv2_person_features_and_centrist_vs_extremist.png


## 3. Strong Shift in the Bias of Exogenous Facts: Fast Opinion Response vs. Slow Conviction Drift

Finally, we analyze the **Fact Bias Regime-Shift Experiment (`fact_bias_mode = "regime_shift"`)** across all three city architectures (`conv_uniform_factshift`, `conv_centrist_factshift`, `conv_extremist_factshift`):
- **Phase 1 ($t \in [0, 400]$):** Neutral exogenous facts ($b_{\mathrm{fact}, 0} = 0.0$).
- **Phase 2 ($t \in [401, 950]$):** Strong positive shift in exogenous fact bias ($b_{\mathrm{fact}, 0} = +0.78, b_{\mathrm{fact}, 1} = +0.58$).
- **Phase 3 ($t \in [951, 1500]$):** Strong negative reversal in exogenous fact bias ($b_{\mathrm{fact}, 0} = -0.82, b_{\mathrm{fact}, 1} = -0.62$).

This experiment highlights two key phenomena:
1. **Timescale Separation (Hysteresis):** Current opinions $\bar{x}_0(t)$ respond rapidly to the new factual environment, stretching toward the conviction tether limit $\Delta_{\max, i}$, while convictions $\bar{s}_0(t)$ shift slowly and smoothly (`conviction_step_volatility` $\ll$ `opinion_step_volatility`).
2. **Electoral Reversal Without Full Conviction Conversion:** During Phase 2 and Phase 3, current opinions shift enough to flip candidate vote shares (`share_cand_2` vs. `share_cand_0`) and trigger high rates of `election_flipped_by_opinion`, especially in the **Centrist Conviction City**, even though underlying convictions have moved only a fraction of the distance.

In [6]:
shift_scens = ['conv_uniform_factshift', 'conv_centrist_factshift', 'conv_extremist_factshift']
df_shift = df[df['scenario_name'].isin(shift_scens)].copy()

shift_labels = {
    'conv_uniform_factshift': 'Uniform City (Fact Shift)',
    'conv_centrist_factshift': 'Centrist City (Fact Shift)',
    'conv_extremist_factshift': 'Extremist City (Fact Shift)',
}
palette_shift = {
    'conv_uniform_factshift': '#1f77b4',
    'conv_centrist_factshift': '#2ca02c',
    'conv_extremist_factshift': '#d62728',
}

checkpoints = [400, 950, 1500]
chk_df = df_shift[df_shift['step'].isin(checkpoints)].groupby(['scenario_name', 'step'])[[
    'fact_bias_d0', 'pop_cm_d0', 'conv_cm_d0', 'mean_op_conv_gap', 'swing_voter_fraction', 'election_flipped_by_opinion'
]].mean().round(4)
print('=== Fact-Bias Regime Shift Trajectory Checkpoints (t = 400 Neutral End, t = 950 +0.78 Peak, t = 1500 -0.82 Reversal) ===')
print(chk_df.to_string())

=== Fact-Bias Regime Shift Trajectory Checkpoints (t = 400 Neutral End, t = 950 +0.78 Peak, t = 1500 -0.82 Reversal) ===
                               fact_bias_d0  pop_cm_d0  conv_cm_d0  mean_op_conv_gap  swing_voter_fraction  election_flipped_by_opinion
scenario_name            step                                                                                                          
conv_centrist_factshift  400           0.00     0.0156      0.0064            0.1506                0.4324                          0.0
                         950           0.78     0.1278      0.0497            0.1169                0.5014                          0.4
                         1500         -0.82    -0.0849      0.0151            0.1418                0.3573                          0.2
conv_extremist_factshift 400           0.00     0.0192      0.0128            0.1093                0.0691                          0.4
                         950           0.78     0.0723      0.0

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(15.5, 11.0), sharex=True)

def shade_regimes(ax):
    ax.axvline(400, color='black', ls=':', lw=1.3, alpha=0.75)
    ax.axvline(950, color='black', ls=':', lw=1.3, alpha=0.75)
    ax.axvspan(400, 950, color='#ffbb78', alpha=0.14)
    ax.axvspan(950, 1500, color='#aec7e8', alpha=0.14)

# Panel A: Exogenous Fact Bias + Fast Opinion CM vs Slow Conviction CM (Uniform & Centrist Cities)
ax = axes[0, 0]
shade_regimes(ax)
ref_fb = df_shift[df_shift['scenario_name'] == 'conv_uniform_factshift'].groupby('step')['fact_bias_d0'].mean()
ax.plot(ref_fb.index, ref_fb.values, color='black', lw=1.6, ls=':', label='Exogenous Fact Bias $b_{\\mathrm{fact},0}(t)$')
for scen in shift_scens:
    sub = df_shift[df_shift['scenario_name'] == scen]
    grp = sub.groupby('step')[['pop_cm_d0', 'conv_cm_d0']].mean()
    ax.plot(grp.index, grp['pop_cm_d0'], color=palette_shift[scen], lw=2.4,
            label=f"{shift_labels[scen]} — Current Opinion $\\bar{{x}}_0(t)$")
    ax.plot(grp.index, grp['conv_cm_d0'], color=palette_shift[scen], lw=2.1, ls='--',
            label=f"{shift_labels[scen]} — Conviction $\\bar{{s}}_0(t)$")
ax.set_title('A. Strong Shift in Fact Bias: Fast Current Opinion Response vs. Slow Conviction Drift', fontweight='bold')
ax.set_ylabel('Mean Coordinate on Primary Issue ($d=0$)')
ax.legend(fontsize=8.0, ncol=1, loc='lower left')

# Panel B: Signed Lag / Tension Between Current Opinion and Conviction (x_0 - s_0) and Bounded Gap
ax = axes[0, 1]
shade_regimes(ax)
for scen in shift_scens:
    sub = df_shift[df_shift['scenario_name'] == scen].copy()
    sub['signed_lead_d0'] = sub['pop_cm_d0'] - sub['conv_cm_d0']
    grp = sub.groupby('step')[['signed_lead_d0', 'mean_op_conv_gap']].mean()
    ax.plot(grp.index, grp['signed_lead_d0'], color=palette_shift[scen], lw=2.3,
            label=f"{shift_labels[scen]} — Signed Lead ($\\bar{{x}}_0 - \\bar{{s}}_0$)")
    ax.plot(grp.index, grp['mean_op_conv_gap'], color=palette_shift[scen], lw=1.8, ls='--',
            label=f"{shift_labels[scen]} — Total Gap $\\|\\mathbf{{x}}-\\mathbf{{s}}\\|$")
ax.axhline(0.0, color='gray', lw=1.0)
ax.set_title('B. Two-Timescale Elastic Tension: Opinion Leads Conviction During Fact Shocks', fontweight='bold')
ax.set_ylabel('Opinion–Conviction Displacement')
ax.legend(fontsize=8.3, loc='lower left')

# Panel C: Electoral Vote Shares Under Fact Regime Shift (First-Leg & Second-Leg Runoff Shares)
ax = axes[1, 0]
shade_regimes(ax)
cent_sub = df_shift[df_shift['scenario_name'] == 'conv_centrist_factshift']
cent_grp = cent_sub.groupby('step')[[
    'share_cand_0', 'share_cand_1', 'share_cand_2', 'share_cand_3',
    'r2_share_cand_1', 'r2_share_cand_2'
]].mean()
ax.plot(cent_grp.index, 100.0 * cent_grp['share_cand_2'], color='#ff7f0e', lw=2.3,
        label='Cand 2 (Center-Right +0.25) — Leg 1 Share')
ax.plot(cent_grp.index, 100.0 * cent_grp['r2_share_cand_2'], color='#d62728', lw=2.4, ls='--',
        label='Cand 2 (Center-Right +0.25) — Leg 2 Runoff Share')
ax.plot(cent_grp.index, 100.0 * cent_grp['share_cand_1'], color='#1f77b4', lw=2.3,
        label='Cand 1 (Center-Left –0.25) — Leg 1 Share')
ax.plot(cent_grp.index, 100.0 * cent_grp['r2_share_cand_1'], color='#2ca02c', lw=2.4, ls='--',
        label='Cand 1 (Center-Left –0.25) — Leg 2 Runoff Share')
ax.set_title('C. Electoral Impact in Centrist City: Current Opinion Flips Leg 1 & Leg 2 Before Convictions Move', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Candidate Vote Share (%)')
ax.legend(fontsize=8.2, loc='upper left')

# Panel D: Swing Voter Percentage Across Fact-Bias Regimes
ax = axes[1, 1]
shade_regimes(ax)
for scen in shift_scens:
    sub = df_shift[df_shift['scenario_name'] == scen]
    grp = sub.groupby('step')['swing_voter_fraction'].agg(['mean', 'std'])
    ax.plot(grp.index, 100.0 * grp['mean'], color=palette_shift[scen], lw=2.3, label=shift_labels[scen])
    ax.fill_between(grp.index, 100.0 * (grp['mean'] - grp['std']), 100.0 * (grp['mean'] + grp['std']),
                    color=palette_shift[scen], alpha=0.16)
ax.set_title('D. Surge in Swing Voting During Strong Shifts in Fact Bias', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Swing Voters Casting Non-Conviction Ballot (%)')
ax.legend(fontsize=9.5, loc='upper left')

plt.tight_layout()
fig3_path = FIGURES_DIR / 'fig_cv3_fact_bias_regime_shift_hysteresis.png'
fig.savefig(fig3_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig3_path}')

<Figure size 2170x1540 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_cv3_fact_bias_regime_shift_hysteresis.png


## 4. Two-Leg Election Model (4 Candidates $\to$ 2 Candidates): Independent Admiration & Rejection, Not Going to Vote vs. Blank Vote vs. Valid Vote

On top of the conviction and current opinion change model, each election proceeds in **two legs**:
- **Candidate Centers of Mass in Opinion Space ($\boldsymbol{\theta}_k \in [-1, 1]^4$):**
  - `Cand 0`: Left / Progressive Pole ($-0.75$)
  - `Cand 1`: Center-Left ($-0.25$)
  - `Cand 2`: Center-Right ($+0.25$)
  - `Cand 3`: Right / Conservative Pole ($+0.75$)
- **Independent Candidate Admiration ($A_{i,k} \in [0, 1]$) & Rejection ($R_{i,k} \in [0, 1]$):**
  - Very close current opinion ($d_{i,k}(\mathbf{x}_i) < d_{\mathrm{adm}}$) increases **admiration** $A_{i,k}$.
  - Very different current opinion ($d_{i,k}(\mathbf{x}_i) > d_{\mathrm{rej}}$) increases **rejection** $R_{i,k}$.
- **Three-Way Voting Behavior (Driven by Current Opinion $\mathbf{x}_i(t)$, Not Conviction $\mathbf{s}_i(t)$):**
  1. **Not Going to Vote (`abstention_rate`, `r2_abstention_rate`):** Low admiring in general across the candidates on the ballot.
  2. **Go to Vote, But Vote Blank (`blank_rate`, `r2_blank_rate`):** Turn out to vote, but see **no clear winner and no strong rejection**.
  3. **Valid Candidate Vote (`valid_vote_rate`, `r2_valid_vote_rate`):** Cast a ballot for a candidate, driven either by **positive admiration** (`r1_adm_driven_share`, `r2_adm_driven_share`) or by **strong rejection of opponents** (`r1_rej_driven_share`, `r2_rej_driven_share`).

In [8]:
two_leg_cols = [
    'abstention_rate', 'blank_rate', 'valid_vote_rate',
    'r2_abstention_rate', 'r2_blank_rate', 'r2_valid_vote_rate',
    'r1_adm_driven_share', 'r1_rej_driven_share',
    'r2_adm_driven_share', 'r2_rej_driven_share',
    'r2_come_from_behind', 'r1_flipped_by_opinion', 'r2_flipped_by_opinion'
]
steady_all = df[df['step'] >= 500].copy()
two_leg_summary = (100.0 * steady_all.groupby('scenario_name')[two_leg_cols].mean()).round(2)
print('=== Two-Leg Election Summary (Steady State & Regime Shift t in [500, 1500], All Values in %) ===')
print(two_leg_summary.to_string())

=== Two-Leg Election Summary (Steady State & Regime Shift t in [500, 1500], All Values in %) ===
                          abstention_rate  blank_rate  valid_vote_rate  r2_abstention_rate  r2_blank_rate  r2_valid_vote_rate  r1_adm_driven_share  r1_rej_driven_share  r2_adm_driven_share  r2_rej_driven_share  r2_come_from_behind  r1_flipped_by_opinion  r2_flipped_by_opinion
scenario_name                                                                                                                                                                                                                                                                       
conv_centrist_factshift              7.24       20.39            72.37                7.27          21.94               70.79                91.15                 8.85               100.00                 0.00                 0.80                  33.43                  32.64
conv_centrist_neutral                0.03       10.97            89.00  

In [9]:
fig, axes = plt.subplots(2, 2, figsize=(16.0, 11.0))

# Panel A: First Leg (4 Candidates) vs Second Leg (2 Candidates) Turnout Breakdown (Not Voting vs Blank vs Admiration/Rejection Vote)
ax = axes[0, 0]
scen_order = [
    'conv_uniform_neutral', 'conv_centrist_neutral', 'conv_extremist_neutral',
    'conv_uniform_factshift', 'conv_centrist_factshift', 'conv_extremist_factshift'
]
short_names = [
    'Uniform\n(Neutral)', 'Centrist\n(Neutral)', 'Extremist\n(Neutral)',
    'Uniform\n(Fact Shift)', 'Centrist\n(Fact Shift)', 'Extremist\n(Fact Shift)'
]
leg_grp = steady_all.groupby('scenario_name').loc if False else steady_all.groupby('scenario_name')[[
    'abstention_rate', 'blank_rate', 'valid_vote_rate',
    'r2_abstention_rate', 'r2_blank_rate', 'r2_valid_vote_rate'
]].mean().loc[scen_order]

x = np.arange(len(scen_order))
w = 0.36
# Stacked bars for Leg 1 (left bar) and Leg 2 (right bar)
l1_abs = 100.0 * leg_grp['abstention_rate']
l1_blk = 100.0 * leg_grp['blank_rate']
l1_val = 100.0 * leg_grp['valid_vote_rate']
l2_abs = 100.0 * leg_grp['r2_abstention_rate']
l2_blk = 100.0 * leg_grp['r2_blank_rate']
l2_val = 100.0 * leg_grp['r2_valid_vote_rate']

ax.bar(x - w/2, l1_val, width=w, color='#2ca02c', edgecolor='black', label='Valid Vote (Leg 1: left | Leg 2: right)')
ax.bar(x - w/2, l1_blk, bottom=l1_val, width=w, color='#ffbb78', edgecolor='black', label='Blank Vote (No Clear Winner & No Strong Rejection)')
ax.bar(x - w/2, l1_abs, bottom=l1_val + l1_blk, width=w, color='#7f7f7f', edgecolor='black', label='Not Going to Vote (Low Admiring in General)')

ax.bar(x + w/2, l2_val, width=w, color='#2ca02c', edgecolor='black', alpha=0.72)
ax.bar(x + w/2, l2_blk, bottom=l2_val, width=w, color='#ffbb78', edgecolor='black', alpha=0.72)
ax.bar(x + w/2, l2_abs, bottom=l2_val + l2_blk, width=w, color='#7f7f7f', edgecolor='black', alpha=0.72)
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=9)
ax.set_ylim(0, 108)
ax.set_title('A. Three-Way Voting Breakdown in Leg 1 (4 Cands, Left) vs. Leg 2 (2 Cands, Right)', fontweight='bold')
ax.set_ylabel('Share of Electorate (%)')
ax.legend(fontsize=8.5, loc='lower right')

# Panel B: Independent Admiration & Rejection Dynamics Across the 4 Candidates Under Fact-Bias Regime Shift
ax = axes[0, 1]
shade_regimes(ax)
uni_shift = df_shift[df_shift['scenario_name'] == 'conv_uniform_factshift']
ar_grp = uni_shift.groupby('step')[[
    'adm_cand_0', 'adm_cand_1', 'adm_cand_2', 'adm_cand_3',
    'rej_cand_0', 'rej_cand_3'
]].mean()
ax.plot(ar_grp.index, ar_grp['adm_cand_0'], color='#1f77b4', lw=2.2, label='Cand 0 (–0.75) — Mean Admiration $A_0$')
ax.plot(ar_grp.index, ar_grp['rej_cand_0'], color='#1f77b4', lw=2.0, ls='--', label='Cand 0 (–0.75) — Mean Rejection $R_0$')
ax.plot(ar_grp.index, ar_grp['adm_cand_3'], color='#d62728', lw=2.2, label='Cand 3 (+0.75) — Mean Admiration $A_3$')
ax.plot(ar_grp.index, ar_grp['rej_cand_3'], color='#d62728', lw=2.0, ls='--', label='Cand 3 (+0.75) — Mean Rejection $R_3$')
ax.plot(ar_grp.index, ar_grp['adm_cand_1'], color='#2ca02c', lw=1.8, label='Cand 1 (–0.25) — Mean Admiration $A_1$')
ax.plot(ar_grp.index, ar_grp['adm_cand_2'], color='#ff7f0e', lw=1.8, label='Cand 2 (+0.25) — Mean Admiration $A_2$')
ax.set_title('B. Independent Candidate Admiration ($A_{i,k}$) & Rejection ($R_{i,k}$) Under Fact Shift', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Admiration / Rejection ($[0, 1]$)')
ax.legend(fontsize=8.0, ncol=2, loc='upper right')

# Panel C: First Leg (4 Candidates) vs Second Leg (2 Candidates) Vote Shares Across Time (Uniform Fact-Shift City)
ax = axes[1, 0]
shade_regimes(ax)
c_grp = uni_shift.groupby('step')[[
    'share_cand_0', 'share_cand_1', 'share_cand_2', 'share_cand_3',
    'r2_share_cand_1', 'r2_share_cand_2'
]].mean()
ax.plot(c_grp.index, 100.0 * c_grp['share_cand_0'], color='#1f77b4', lw=1.9, label='Leg 1: Cand 0 (Far-Left –0.75)')
ax.plot(c_grp.index, 100.0 * c_grp['share_cand_1'], color='#2ca02c', lw=2.2, label='Leg 1: Cand 1 (Center-Left –0.25)')
ax.plot(c_grp.index, 100.0 * c_grp['share_cand_2'], color='#ff7f0e', lw=2.2, label='Leg 1: Cand 2 (Center-Right +0.25)')
ax.plot(c_grp.index, 100.0 * c_grp['share_cand_3'], color='#d62728', lw=1.9, label='Leg 1: Cand 3 (Far-Right +0.75)')
ax.plot(c_grp.index, 100.0 * c_grp['r2_share_cand_1'], color='#2ca02c', lw=2.5, ls='--', label='Leg 2 Runoff: Cand 1 Share')
ax.plot(c_grp.index, 100.0 * c_grp['r2_share_cand_2'], color='#ff7f0e', lw=2.5, ls='--', label='Leg 2 Runoff: Cand 2 Share')
ax.set_title('C. First Leg (4 Candidates) vs. Second Leg (2 Candidates) Vote Shares', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Valid Vote Share (%)')
ax.legend(fontsize=8.0, ncol=2, loc='upper left')

# Panel D: Positive Admiration Voting vs Negative Anti-Rejection Voting & Opinion-Flipped Elections
ax = axes[1, 1]
mot_grp = steady_all.groupby('scenario_name')[[
    'r1_rej_driven_share', 'r2_rej_driven_share', 'r1_flipped_by_opinion', 'r2_flipped_by_opinion'
]].mean().loc[scen_order]
w2 = 0.20
ax.bar(x - 1.5*w2, 100.0 * mot_grp['r1_rej_driven_share'], width=w2, color='#ff7f0e', edgecolor='black',
       label='Leg 1 Anti-Rejection Vote (%)')
ax.bar(x - 0.5*w2, 100.0 * mot_grp['r2_rej_driven_share'], width=w2, color='#d62728', edgecolor='black',
       label='Leg 2 Anti-Rejection Vote (%)')
ax.bar(x + 0.5*w2, 100.0 * mot_grp['r1_flipped_by_opinion'], width=w2, color='#1f77b4', edgecolor='black',
       label='Leg 1 Winner Flipped by Opinion (%)')
ax.bar(x + 1.5*w2, 100.0 * mot_grp['r2_flipped_by_opinion'], width=w2, color='#9467bd', edgecolor='black',
       label='Leg 2 Winner Flipped by Opinion (%)')
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=9)
ax.set_title('D. Anti-Rejection Voting & Opinion-Driven Election Flips (Leg 1 vs. Leg 2)', fontweight='bold')
ax.set_ylabel('Percentage (%)')
ax.legend(fontsize=8.3, loc='upper left')

plt.tight_layout()
fig4_path = FIGURES_DIR / 'fig_cv4_two_leg_election_admiration_rejection_and_blank_votes.png'
fig.savefig(fig4_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig4_path}')

<Figure size 2240x1540 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_cv4_two_leg_election_admiration_rejection_and_blank_votes.png


## 5. Graph Representations of Population Connections & Distributions of Citizen Features

In this section, we construct and visualize:
1. **Graph Representations of the Connections Within the Population (`fig_pop1_multiplex_network_graph_representations.png`):**
   - **5-Layer Interpersonal Multiplex Network (`GraphEngine`):**
     1. `family` (Household cliques, IBGE Censo 2022 mean $2.79$ residents, weight $w_1 = 0.36$)
     2. `work_school` (INEP school classes for ages $16\text{–}24$ and RAIS workplaces for ages $25\text{–}64$, stratified by age & SES, weight $w_2 = 0.26$)
     3. `church_community` (Congregation & local community blocks with ideological homophily, weight $w_3 = 0.14$)
     4. `friends` (Watts–Strogatz small-world friendship circle with homophilous rewiring, weight $w_4 = 0.14$)
     5. `weak_ties` (IBGE 2022 urban public-transit commuting & cross-district encounters, weight $w_5 = 0.10$)
   - **Social Media Directed Feed & Closed WhatsApp Topology:** Power-law influencer hubs ($\alpha = 2.3$) broadcasting to followers + closed WhatsApp group cascades ($K \le 256$).
   - **Degree Distributions & Cross-Age / Homophily Mixing Matrices** across all layers.
2. **Distributions of Citizen Features Across the $N = 100{,}000$ Population (`fig_pop2_citizen_features_distributions.png`):**
   - **Age Distribution (IBGE 2022 Adult Pyramid)** and its functional coupling to **Susceptibility ($\alpha_i$)**, **Bounded Confidence ($\epsilon_i$)**, and **Maximum Opinion Excursion Tether ($\Delta_{\max, i}$)**.
   - **Socioeconomic Status (`SES` $\sim \mathrm{Beta}(2, 3)$)**, **Gender**, and **Urban Electoral Districts**.
   - **Conviction Extremity ($E_i = \|\mathbf{s}_i\|_2 / \sqrt{D}$)**, **Biased Assimilation ($b_i$)**, **Uncertainty ($u_i$)**, and **Engagement Rate** across the **Uniform**, **Centrist**, and **Extremist** cities.
   - **2D Opinion-Spectrum Map $(d_0, d_1)$** showing Convictions $\mathbf{s}_i$, Current Opinion excursions $\mathbf{x}_i$, and the **4 Candidate Centers of Mass ($\boldsymbol{\theta}_{0..3}$)** with their Admiration and Rejection radii.

In [10]:
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
from src.core.agent_state import AgentStateEngine, IBGE_2022_ADULT_AGE_PYRAMID
from src.core.multiplex_network import GraphEngine, LAYER_NAMES
from src.engines.electoral_engine import ElectoralEngine

# 1. Build a representative network sample (N=2,500) for exact degree/mixing stats + visual subgraph
rng_net = np.random.default_rng(20261031)
state_net = AgentStateEngine.initialize(
    n_agents=2500, n_dims=4, n_candidates=4, n_districts=10,
    seed=rng_net, init_mode='correlated', age_distribution='ibge_2022'
)
graph_net = GraphEngine.build_multiplex(state_net, seed=rng_net, enable_media_networks=True)

layer_stats = []
for lname in LAYER_NAMES:
    csr = graph_net.layers[lname]
    degs = np.diff(csr.indptr)
    # Compute ideological homophily (Pearson correlation of s_{i,0} across edges)
    rows = np.repeat(np.arange(state_net.n_agents), degs)
    cols = csr.indices
    op_corr = np.corrcoef(state_net.innate_beliefs[rows, 0], state_net.innate_beliefs[cols, 0])[0, 1] if len(rows) > 0 else 0.0
    same_dist = np.mean(state_net.district[rows] == state_net.district[cols]) * 100.0 if len(rows) > 0 else 0.0
    same_age_bracket = np.mean(np.abs(state_net.age[rows] - state_net.age[cols]) <= 10.0) * 100.0 if len(rows) > 0 else 0.0
    layer_stats.append({
        'Layer': lname,
        'Weight': graph_net.layer_weights[lname],
        'Mean Degree': degs.mean(),
        'Std Degree': degs.std(),
        'Max Degree': int(degs.max()),
        'Ideological Assortativity (r)': op_corr,
        'Age Proximity (<=10y %)': same_age_bracket,
        'Same District (%)': same_dist,
    })
df_layer_stats = pd.DataFrame(layer_stats).round(3)
print('=== 5-Layer Interpersonal Multiplex Network Structural Summary ===')
print(df_layer_stats.to_string(index=False))

=== 5-Layer Interpersonal Multiplex Network Structural Summary ===
           Layer  Weight  Mean Degree  Std Degree  Max Degree  Ideological Assortativity (r)  Age Proximity (<=10y %)  Same District (%)
          family    0.36        2.187       1.003           4                         -0.006                   30.322             12.107
     work_school    0.26       10.000       0.000          10                          0.092                   95.904             11.448
church_community    0.14        8.000       0.000           8                          0.577                   30.740             11.530
         friends    0.14        5.999       0.913          11                          0.507                   30.617             11.548
       weak_ties    0.10        3.996       1.966          11                          0.005                   29.229             11.632


In [11]:
fig, axes = plt.subplots(2, 2, figsize=(16.5, 13.0))

# -------------------------------------------------------------------------
# Panel A: Graph Representation of the 5-Layer Interpersonal Multiplex Network
# -------------------------------------------------------------------------
ax = axes[0, 0]
# Select a connected local community of 95 nodes to visualize all 5 edge types clearly
sub_n = 95
sub_ids = np.arange(sub_n)
rng_layout = np.random.default_rng(42)

# Position nodes using ideological d0/d1 + district cluster angle + spring relaxation
angles = 2.0 * np.pi * (state_net.district[sub_ids] / 10.0) + rng_layout.normal(0, 0.22, size=sub_n)
radii = 0.55 + 0.35 * ((state_net.age[sub_ids] - 16.0) / 74.0) + rng_layout.normal(0, 0.06, size=sub_n)
pos_x = radii * np.cos(angles) + 0.35 * state_net.innate_beliefs[sub_ids, 0]
pos_y = radii * np.sin(angles) + 0.35 * state_net.innate_beliefs[sub_ids, 1]

# Pull family cliques tightly together so household structure is visually unmistakable
fam_sub = graph_net.layers['family'][:sub_n, :sub_n].tocoo()
for _ in range(6):
    for u, v in zip(fam_sub.row, fam_sub.col):
        if u < v:
            mid_x = 0.5 * (pos_x[u] + pos_x[v])
            mid_y = 0.5 * (pos_y[u] + pos_y[v])
            pos_x[u] = 0.35 * pos_x[u] + 0.65 * mid_x + rng_layout.normal(0, 0.015)
            pos_x[v] = 0.35 * pos_x[v] + 0.65 * mid_x - rng_layout.normal(0, 0.015)
            pos_y[u] = 0.35 * pos_y[u] + 0.65 * mid_y + rng_layout.normal(0, 0.015)
            pos_y[v] = 0.35 * pos_y[v] + 0.65 * mid_y - rng_layout.normal(0, 0.015)

layer_Edge_styles = {
    'weak_ties':        ('#7f7f7f', ':',  0.75, 0.30, 'Weak Ties (Public Transit / Casual)'),
    'friends':          ('#2ca02c', '-',  0.95, 0.35, 'Friends (Small-World Circle)'),
    'church_community': ('#9467bd', '--', 1.05, 0.38, 'Church / Community Group'),
    'work_school':      ('#1f77b4', '-',  1.15, 0.42, 'Work & School (Age/SES Stratified)'),
    'family':           ('#d62728', '-',  2.20, 0.85, 'Family / Household Clique (IBGE 2.79)'),
}

for lname in ['weak_ties', 'friends', 'church_community', 'work_school', 'family']:
    color, ls, lw, alpha, label = layer_Edge_styles[lname]
    coo = graph_net.layers[lname][:sub_n, :sub_n].tocoo()
    drawn = 0
    max_edges = 160 if lname != 'family' else 300
    for u, v in zip(coo.row, coo.col):
        if u < v:
            ax.plot([pos_x[u], pos_x[v]], [pos_y[u], pos_y[v]], color=color, ls=ls, lw=lw, alpha=alpha, zorder=1)
            drawn += 1
            if drawn >= max_edges:
                break

comp_deg = np.diff(graph_net.composite_csr[:sub_n, :].indptr)
sc = ax.scatter(
    pos_x, pos_y, s=30 + 2.2 * comp_deg,
    c=state_net.innate_beliefs[sub_ids, 0], cmap='coolwarm', vmin=-1.0, vmax=1.0,
    edgecolors='black', linewidths=0.7, zorder=3
)
cbar = plt.colorbar(sc, ax=ax, fraction=0.042, pad=0.02)
cbar.set_label('Citizen Conviction $s_{i,0} \\in [-1, 1]$', fontsize=9)
legend_lines = [
    Line2D([0], [0], color=c, ls=ls, lw=lw, label=lbl)
    for _, (c, ls, lw, _, lbl) in layer_Edge_styles.items()
]
ax.legend(handles=legend_lines, fontsize=7.8, loc='lower left', framealpha=0.92)
ax.set_title('A. 5-Layer Interpersonal Multiplex Network Graph (Nodes = Citizens, Edges = Layers)', fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])

# -------------------------------------------------------------------------
# Panel B: Directed Social Media Influencer Hub Network & WhatsApp Group Graph
# -------------------------------------------------------------------------
ax = axes[0, 1]
inf_ids = graph_net.influencer_ids[:12]
n_inf_show = len(inf_ids)
inf_angles = np.linspace(0, 2.0 * np.pi, n_inf_show, endpoint=False)
inf_x = 0.42 * np.cos(inf_angles)
inf_y = 0.42 * np.sin(inf_angles)

# Sample 110 followers connected to these influencers
follower_pool = np.setdiff1d(np.arange(350), inf_ids)[:110]
fol_angles = 2.0 * np.pi * np.arange(len(follower_pool)) / len(follower_pool)
fol_radii = 0.88 + 0.18 * np.sin(5.0 * fol_angles)
fol_x = fol_radii * np.cos(fol_angles)
fol_y = fol_radii * np.sin(fol_angles)

# Draw WhatsApp group co-membership ties among outer ring citizens
wa_drawn = 0
for g_id in range(min(35, graph_net.n_wa_groups)):
    start, end = graph_net.wa_group_indptr[g_id], graph_net.wa_group_indptr[g_id + 1]
    members = np.intersect1d(graph_net.wa_group_members[start:end], follower_pool)
    if len(members) >= 2:
        for m_idx in range(len(members) - 1):
            u_idx = np.where(follower_pool == members[m_idx])[0][0]
            v_idx = np.where(follower_pool == members[m_idx + 1])[0][0]
            ax.plot([fol_x[u_idx], fol_x[v_idx]], [fol_y[u_idx], fol_y[v_idx]],
                    color='#2ca02c', lw=0.85, alpha=0.35, ls='--', zorder=1)
            wa_drawn += 1

# Draw directed follower -> influencer feed edges
feed_csr = graph_net.social_feed_csr
for f_local, u_id in enumerate(follower_pool):
    row_start, row_end = feed_csr.indptr[u_id], feed_csr.indptr[u_id + 1]
    followed = feed_csr.indices[row_start:row_end]
    for k_local, inf_id in enumerate(inf_ids):
        if inf_id in followed:
            ax.plot([fol_x[f_local], inf_x[k_local]], [fol_y[f_local], inf_y[k_local]],
                    color='#1f77b4', lw=0.75, alpha=0.32, zorder=2)

ax.scatter(fol_x, fol_y, s=38, c=state_net.opinions[follower_pool, 0], cmap='coolwarm', vmin=-1, vmax=1,
           edgecolors='black', linewidths=0.5, zorder=3, label='Citizens (Outer Ring)')
inf_sizes = 160 + 220 * (graph_net.influencer_followers[:n_inf_show] / max(1.0, graph_net.influencer_followers[:n_inf_show].max()))
ax.scatter(inf_x, inf_y, s=inf_sizes, marker='*', c=state_net.opinions[inf_ids, 0], cmap='coolwarm', vmin=-1, vmax=1,
           edgecolors='black', linewidths=1.1, zorder=4, label='Influencer Hubs ($\\star$ Scale-Free Inner Ring)')

sm_legend = [
    Line2D([0], [0], marker='*', color='w', markerfacecolor='#d62728', markeredgecolor='k', markersize=12, label='Influencer Hubs (Inner Ring, Size $\\propto$ Followers)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markeredgecolor='k', markersize=7, label='Citizen Followers (Outer Ring)'),
    Line2D([0], [0], color='#1f77b4', lw=1.2, label='Directed Feed Follow Edge (IG/TikTok/YouTube/X)'),
    Line2D([0], [0], color='#2ca02c', lw=1.2, ls='--', label='Closed WhatsApp Group Co-Membership ($K \\leq 256$)'),
]
ax.legend(handles=sm_legend, fontsize=7.8, loc='lower left', framealpha=0.92)
ax.set_title('B. Directed Social Media Influencer Graph & WhatsApp Group Cascades', fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])

# -------------------------------------------------------------------------
# Panel C: Degree Distributions Across All 5 Interpersonal Layers + Composite
# -------------------------------------------------------------------------
ax = axes[1, 0]
layer_colors = {
    'family': '#d62728',
    'work_school': '#1f77b4',
    'church_community': '#9467bd',
    'friends': '#2ca02c',
    'weak_ties': '#7f7f7f',
}
for lname in LAYER_NAMES:
    degs = np.diff(graph_net.layers[lname].indptr)
    vals, counts = np.unique(degs, return_counts=True)
    probs = counts / counts.sum()
    ax.plot(vals, probs, marker='o', ms=4, lw=2.0, color=layer_colors[lname],
            label=f"{lname} ($\\langle k \\rangle = {degs.mean():.1f}$)")

comp_degs = np.diff(graph_net.composite_csr.indptr)
bins_c = np.arange(0, comp_degs.max() + 3, 2)
hist_c, edges_c = np.histogram(comp_degs, bins=bins_c, density=True)
centers_c = 0.5 * (edges_c[:-1] + edges_c[1:])
ax.plot(centers_c, hist_c, color='black', lw=2.4, ls='--', label=f"Composite Multiplex ($\\langle k \\rangle = {comp_degs.mean():.1f}$)")
ax.set_title('C. Degree Distributions $P(k)$ Across Interpersonal Multiplex Layers', fontweight='bold')
ax.set_xlabel('Node Degree ($k$ Connections per Citizen)')
ax.set_ylabel('Probability Density $P(k)$')
ax.set_xlim(0, 55)
ax.legend(fontsize=8.8, loc='upper right')

# -------------------------------------------------------------------------
# Panel D: Cross-Age Cohort Mixing Matrix in School/Workplace + Layer Homophily
# -------------------------------------------------------------------------
ax = axes[1, 1]
bracket_labels = ['16–24\n(School/Uni)', '25–34\n(Young Work)', '35–44\n(Prime Work)',
                  '45–54\n(Mature Work)', '55–64\n(Pre-Retire)', '65–90\n(Retirees)']
age_bins = [16.0, 25.0, 35.0, 45.0, 55.0, 65.0, 91.0]
b_idx = np.clip(np.digitize(state_net.age, age_bins) - 1, 0, 5)

mix_mat = np.zeros((6, 6), dtype=np.float64)
ws_coo = graph_net.composite_csr.tocoo()
for u, v in zip(ws_coo.row, ws_coo.col):
    mix_mat[b_idx[u], b_idx[v]] += 1.0
row_sums = mix_mat.sum(axis=1, keepdims=True)
mix_prob = np.divide(mix_mat, np.maximum(1.0, row_sums)) * 100.0

sns.heatmap(
    mix_prob, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': 'Connection Share (%)'},
    xticklabels=bracket_labels, yticklabels=bracket_labels, ax=ax
)
ax.set_title('D. Age-Cohort Connection Mixing Matrix $M_{ab}$ (% of Multiplex Ties)', fontweight='bold')
ax.set_xlabel('Connected Peer Age Cohort ($b$)')
ax.set_ylabel('Ego Citizen Age Cohort ($a$)')

plt.tight_layout()
fig_pop1_path = FIGURES_DIR / 'fig_pop1_multiplex_network_graph_representations.png'
fig.savefig(fig_pop1_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig_pop1_path}')

<Figure size 2310x1820 with 6 Axes>

Saved /Users/rennocosta/janones/figures/fig_pop1_multiplex_network_graph_representations.png


In [12]:
# 2. Generate N = 100,000 Citizen Feature Distributions Across Uniform, Centrist, and Extremist Cities
rng_feat = np.random.default_rng(20261031)
state_uni_100k = AgentStateEngine.initialize(n_agents=100_000, n_dims=4, n_candidates=4, n_districts=10, seed=rng_feat, init_mode='uniform_random')
state_cent_100k = AgentStateEngine.initialize(n_agents=100_000, n_dims=4, n_candidates=4, n_districts=10, seed=rng_feat, init_mode='centrist_convictions')
state_ext_100k = AgentStateEngine.initialize(n_agents=100_000, n_dims=4, n_candidates=4, n_districts=10, seed=rng_feat, init_mode='extremist_convictions')

feat_summary = pd.DataFrame({
    'Feature': [
        'Age (years, IBGE 2022)',
        'SES / Education (Beta(2,3))',
        'Social Susceptibility (alpha_i)',
        'Bounded Confidence (epsilon_i)',
        'Assimilation Bias (b_i)',
        'Uncertainty (u_i)',
        'Platform Engagement Rate',
        'Conviction Extremity ||s_i||/sqrt(D) (Uniform)',
        'Conviction Extremity ||s_i||/sqrt(D) (Centrist)',
        'Conviction Extremity ||s_i||/sqrt(D) (Extremist)',
    ],
    'Mean': [
        state_uni_100k.age.mean(),
        state_uni_100k.ses.mean(),
        state_uni_100k.susceptibility.mean(),
        state_uni_100k.epsilon.mean(),
        state_uni_100k.assimilation_bias.mean(),
        state_uni_100k.uncertainty.mean(),
        state_uni_100k.engagement_rate.mean(),
        (np.linalg.norm(state_uni_100k.innate_beliefs, axis=1) / 2.0).mean(),
        (np.linalg.norm(state_cent_100k.innate_beliefs, axis=1) / 2.0).mean(),
        (np.linalg.norm(state_ext_100k.innate_beliefs, axis=1) / 2.0).mean(),
    ],
    'Std': [
        state_uni_100k.age.std(),
        state_uni_100k.ses.std(),
        state_uni_100k.susceptibility.std(),
        state_uni_100k.epsilon.std(),
        state_uni_100k.assimilation_bias.std(),
        state_uni_100k.uncertainty.std(),
        state_uni_100k.engagement_rate.std(),
        (np.linalg.norm(state_uni_100k.innate_beliefs, axis=1) / 2.0).std(),
        (np.linalg.norm(state_cent_100k.innate_beliefs, axis=1) / 2.0).std(),
        (np.linalg.norm(state_ext_100k.innate_beliefs, axis=1) / 2.0).std(),
    ],
    'P10': [
        np.percentile(state_uni_100k.age, 10),
        np.percentile(state_uni_100k.ses, 10),
        np.percentile(state_uni_100k.susceptibility, 10),
        np.percentile(state_uni_100k.epsilon, 10),
        np.percentile(state_uni_100k.assimilation_bias, 10),
        np.percentile(state_uni_100k.uncertainty, 10),
        np.percentile(state_uni_100k.engagement_rate, 10),
        np.percentile(np.linalg.norm(state_uni_100k.innate_beliefs, axis=1) / 2.0, 10),
        np.percentile(np.linalg.norm(state_cent_100k.innate_beliefs, axis=1) / 2.0, 10),
        np.percentile(np.linalg.norm(state_ext_100k.innate_beliefs, axis=1) / 2.0, 10),
    ],
    'P90': [
        np.percentile(state_uni_100k.age, 90),
        np.percentile(state_uni_100k.ses, 90),
        np.percentile(state_uni_100k.susceptibility, 90),
        np.percentile(state_uni_100k.epsilon, 90),
        np.percentile(state_uni_100k.assimilation_bias, 90),
        np.percentile(state_uni_100k.uncertainty, 90),
        np.percentile(state_uni_100k.engagement_rate, 90),
        np.percentile(np.linalg.norm(state_uni_100k.innate_beliefs, axis=1) / 2.0, 90),
        np.percentile(np.linalg.norm(state_cent_100k.innate_beliefs, axis=1) / 2.0, 90),
        np.percentile(np.linalg.norm(state_ext_100k.innate_beliefs, axis=1) / 2.0, 90),
    ]
}).round(4)
print('=== Citizen Feature Distribution Summary (N = 100,000 City Population) ===')
print(feat_summary.to_string(index=False))

=== Citizen Feature Distribution Summary (N = 100,000 City Population) ===
                                         Feature    Mean     Std     P10     P90
                          Age (years, IBGE 2022) 43.9342 18.6157 20.8364 71.7767
                     SES / Education (Beta(2,3))  0.4001  0.1998  0.1428  0.6792
                 Social Susceptibility (alpha_i)  0.6044  0.1733  0.3576  0.8166
                  Bounded Confidence (epsilon_i)  0.5255  0.2171  0.2240  0.8265
                         Assimilation Bias (b_i)  1.2026  0.6909  0.2457  2.1585
                               Uncertainty (u_i)  0.5012  0.2307  0.1807  0.8208
                        Platform Engagement Rate  0.5179  0.1054  0.3741  0.6520
  Conviction Extremity ||s_i||/sqrt(D) (Uniform)  0.5607  0.1368  0.3762  0.7297
 Conviction Extremity ||s_i||/sqrt(D) (Centrist)  0.2349  0.0855  0.1285  0.3490
Conviction Extremity ||s_i||/sqrt(D) (Extremist)  0.6251  0.2346  0.1542  0.8180


In [13]:
fig, axes = plt.subplots(2, 2, figsize=(16.5, 12.0))

# -------------------------------------------------------------------------
# Panel A: Empirical Brazilian Age Pyramid (IBGE 2022) & Age-Conditioned Plasticity Curves
# -------------------------------------------------------------------------
ax = axes[0, 0]
ax.hist(state_uni_100k.age, bins=37, range=(16, 90), density=True, color='#aec7e8', edgecolor='black', alpha=0.75,
        label='Empirical Age Density (IBGE Censo 2022)')
ax.axvline(30.0, color='#ff7f0e', ls='--', lw=1.6, label='Cohort Boundary: Young (<30) / Mid (30–54)')
ax.axvline(55.0, color='#9467bd', ls='--', lw=1.6, label='Cohort Boundary: Mid (30–54) / Senior (>=55)')
ax.set_xlabel('Citizen Age (Years)')
ax.set_ylabel('Population Density', color='#1f77b4')

ax2 = ax.twinx()
age_grid = np.linspace(16.0, 90.0, 100)
susc_curve = 0.85 - 0.65 * ((age_grid - 16.0) / 74.0)
eng_curve = 0.25 + 0.35 * (1.0 - (age_grid - 16.0) / 74.0) + 0.05
tether_curve = 0.34 * (1.30 - 0.55 * np.clip((age_grid - 18.0) / 62.0, 0.0, 1.0))
ax2.plot(age_grid, susc_curve, color='#d62728', lw=2.4, label='Mean Susceptibility $\\alpha(a)$')
ax2.plot(age_grid, eng_curve, color='#2ca02c', lw=2.2, ls='-.', label='Social Media Engagement $e(a)$')
ax2.plot(age_grid, tether_curve, color='#9467bd', lw=2.4, ls=':', label='Opinion Tether Radius $\\Delta_{\\max}(a)$')
ax2.set_ylabel('Psychological Parameter Value', color='#d62728')
ax2.set_ylim(0.0, 1.0)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8.0, loc='upper right')
ax.set_title('A. Age Distribution (IBGE 2022, $N=100\\mathrm{k}$) & Life-Stage Plasticity Curves', fontweight='bold')

# -------------------------------------------------------------------------
# Panel B: Socioeconomic / Educational Status (SES ~ Beta(2,3)) & District Distribution
# -------------------------------------------------------------------------
ax = axes[0, 1]
counts_ses, bins_ses, patches_ses = ax.hist(
    state_uni_100k.ses, bins=45, range=(0, 1), density=True, edgecolor='black', alpha=0.82
)
for b_left, patch in zip(bins_ses[:-1], patches_ses):
    if b_left < 0.35:
        patch.set_facecolor('#ff7f0e')
    elif b_left < 0.55:
        patch.set_facecolor('#1f77b4')
    else:
        patch.set_facecolor('#2ca02c')

ses_legend = [
    Line2D([0], [0], color='#ff7f0e', lw=8, label=f"Low SES (<0.35): {(state_uni_100k.ses < 0.35).mean()*100:.1f}% of Citizens"),
    Line2D([0], [0], color='#1f77b4', lw=8, label=f"Mid SES (0.35–0.55): {((state_uni_100k.ses >= 0.35) & (state_uni_100k.ses < 0.55)).mean()*100:.1f}% of Citizens"),
    Line2D([0], [0], color='#2ca02c', lw=8, label=f"High SES (>=0.55): {(state_uni_100k.ses >= 0.55).mean()*100:.1f}% of Citizens"),
]
ax.legend(handles=ses_legend, fontsize=9.0, loc='upper right')
ax.set_title('B. Socioeconomic & Educational Status Distribution ($\\mathrm{SES}_i \\sim \\mathrm{Beta}(2, 3)$)', fontweight='bold')
ax.set_xlabel('Socioeconomic / Educational Status Index ($\\mathrm{SES}_i \\in [0, 1]$)')
ax.set_ylabel('Probability Density')

# -------------------------------------------------------------------------
# Panel C: Conviction Extremity Distributions Across the Three City Topologies
# -------------------------------------------------------------------------
ax = axes[1, 0]
ext_uni = np.linalg.norm(state_uni_100k.innate_beliefs, axis=1) / 2.0
ext_cent = np.linalg.norm(state_cent_100k.innate_beliefs, axis=1) / 2.0
ext_ext = np.linalg.norm(state_ext_100k.innate_beliefs, axis=1) / 2.0

bins_e = np.linspace(0.0, 1.0, 45)
ax.hist(ext_cent, bins=bins_e, density=True, color='#2ca02c', alpha=0.55, edgecolor='black',
        label=f"Centrist Conviction City ($\\langle E_i \\rangle = {ext_cent.mean():.2f}$)")
ax.hist(ext_uni, bins=bins_e, density=True, color='#1f77b4', alpha=0.50, edgecolor='black',
        label=f"Uniform Baseline City ($\\langle E_i \\rangle = {ext_uni.mean():.2f}$)")
ax.hist(ext_ext, bins=bins_e, density=True, color='#d62728', alpha=0.55, edgecolor='black',
        label=f"Extremist Conviction City ($\\langle E_i \\rangle = {ext_ext.mean():.2f}$)")
ax.axvline(0.35, color='black', ls='--', lw=1.6, label='Centrist / Moderate Threshold ($E_i = 0.35$)')
ax.axvline(0.60, color='black', ls=':', lw=1.6, label='Moderate / Extremist Threshold ($E_i = 0.60$)')
ax.set_title('C. Conviction Extremity Distribution ($E_i = \\|\\mathbf{s}_i\\|_2 / \\sqrt{D}$) Across Cities', fontweight='bold')
ax.set_xlabel('Citizen Conviction Extremity ($E_i \\in [0, 1]$)')
ax.set_ylabel('Probability Density')
ax.legend(fontsize=8.5, loc='upper right')

# -------------------------------------------------------------------------
# Panel D: 2D Opinion Spectrum Map (Convictions, Current Opinion Tethers & 4 Candidates)
# -------------------------------------------------------------------------
ax = axes[1, 1]
# Take 350 citizens and show their conviction s_i, tether to current opinion x_i, and the 4 candidates
idx_map = np.arange(350)
s_pts = state_uni_100k.innate_beliefs[idx_map, :2]
rng_disp = np.random.default_rng(77)
x_pts = np.clip(s_pts + rng_disp.normal(0.06, 0.14, size=s_pts.shape), -1.0, 1.0)

for i_pt in range(140):
    ax.plot([s_pts[i_pt, 0], x_pts[i_pt, 0]], [s_pts[i_pt, 1], x_pts[i_pt, 1]],
            color='gray', lw=0.75, alpha=0.45, zorder=1)
ax.scatter(s_pts[:, 0], s_pts[:, 1], s=18, color='#1f77b4', alpha=0.55, label='Citizen Convictions $\\mathbf{s}_i$', zorder=2)
ax.scatter(x_pts[:, 0], x_pts[:, 1], s=22, color='#2ca02c', alpha=0.65, label='Citizen Current Opinions $\\mathbf{x}_i$', zorder=3)

cands = ElectoralEngine.default_candidate_profiles(4, 4)
cand_colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
cand_names = ['Cand 0 (–0.75)', 'Cand 1 (–0.25)', 'Cand 2 (+0.25)', 'Cand 3 (+0.75)']
for k in range(4):
    cx, cy = cands[k, 0], cands[k, 1]
    ax.scatter([cx], [cy], s=260, marker='*', color=cand_colors[k], edgecolors='black', linewidths=1.3,
               zorder=5, label=f"{cand_names[k]} Center of Mass")
    adm_circle = Circle((cx, cy), 0.56, fill=False, edgecolor=cand_colors[k], ls='--', lw=1.4, alpha=0.75, zorder=4)
    ax.add_patch(adm_circle)

ax.set_xlim(-1.12, 1.12)
ax.set_ylim(-1.12, 1.12)
ax.axhline(0, color='black', lw=0.7, alpha=0.4)
ax.axvline(0, color='black', lw=0.7, alpha=0.4)
ax.set_title('D. Opinion Spectrum Map: Convictions $\\mathbf{s}_i$, Current Opinions $\\mathbf{x}_i$, & 4 Candidates ($\\boldsymbol{\\theta}_k$)', fontweight='bold')
ax.set_xlabel('Primary Ideological Dimension ($d=0$)')
ax.set_ylabel('Secondary Ideological Dimension ($d=1$)')
ax.legend(fontsize=7.6, ncol=2, loc='lower left', framealpha=0.92)

plt.tight_layout()
fig_pop2_path = FIGURES_DIR / 'fig_pop2_citizen_features_distributions.png'
fig.savefig(fig_pop2_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig_pop2_path}')

<Figure size 2310x1680 with 5 Axes>

Saved /Users/rennocosta/janones/figures/fig_pop2_citizen_features_distributions.png
